# SHORE — Interactive Visualisation

This notebook generates a unit-sphere test case from scratch, then inspects
the resulting `.geo` volume grid and `.npz` surface layer using PyVista's HTML
backend, which renders inline as a WebGL canvas — no display or X server required.

**Prerequisites**
```bash
pip install shore[vis]   # adds pyvista
```

In [ ]:
import numpy as np
import pyvista as pv

from shore.io.geo import read_geo, write_geo
from shore.io.vtk import geo_to_pyvista, surface_to_pyvista, write_surface_vts, write_vts
from shore.vis import view_geo, view_surface
from shore.volume.jacobian import jacobian_field, jacobian_per_layer

# Use the HTML (WebGL) backend for inline rendering
pv.set_jupyter_backend('html')

## 0 — Generate a unit-sphere test case

Creates `sphere.stl`, projects it onto a structured `(ni×nj)` surface layer
saved as `sphere_k0.npz`, then marches `nk` wall-normal layers to produce
`sphere.geo`. Skips any step whose output file already exists.

In [ ]:
from pathlib import Path

import trimesh

from shore.surface.io import load_stl
from shore.surface.remesh import project_sphere_to_surface
from shore.volume.hyperbolic import march

NI, NJ, NK = 40, 60, 20
DS, GROWTH = 0.02, 1.10

STL_PATH = Path('sphere.stl')
NPZ_PATH = Path('sphere_k0.npz')
GEO_PATH = Path('sphere.geo')

# ---- Step 1: STL ----
if not STL_PATH.exists():
    m = trimesh.creation.icosphere(subdivisions=4, radius=1.0)
    m.export(str(STL_PATH))
    print(f'Generated {STL_PATH}  ({len(m.faces)} triangles)')
else:
    print(f'{STL_PATH} already exists, skipping.')

# ---- Step 2: surface projection ----
if not NPZ_PATH.exists():
    mesh = load_stl(STL_PATH)
    k0 = project_sphere_to_surface(mesh, ni=NI, nj=NJ)
    np.savez_compressed(str(NPZ_PATH), k0=k0)
    print(f'Generated {NPZ_PATH}  shape={k0.shape}')
else:
    print(f'{NPZ_PATH} already exists, skipping.')
    k0 = np.load(str(NPZ_PATH))['k0']

# ---- Step 3: hyperbolic marching ----
if not GEO_PATH.exists():
    layers_done = []

    def _cb(k: int, jmin: float, step: float) -> None:
        layers_done.append(k)

    grid = march(k0, nk=NK, ds=DS, growth=GROWTH, layer_callback=_cb)
    write_geo(str(GEO_PATH), grid)
    print(f'Generated {GEO_PATH}  shape={grid.shape[:3]}  layers={len(layers_done)}')
else:
    print(f'{GEO_PATH} already exists, skipping.')
    grid = read_geo(GEO_PATH)

## 1 — Load and inspect the volume grid

In [ ]:
grid = read_geo(GEO_PATH)
ni, nj, nk, _ = grid.shape
print(f'Grid dimensions : ni={ni}  nj={nj}  nk={nk}')
print(f'Points          : {ni*nj*nk:,}')
print(f'Cells           : {(ni-1)*nj*(nk-1):,}')

xyz = grid.reshape(-1, 3)
print(f'x bounds : [{xyz[:,0].min():.4f},  {xyz[:,0].max():.4f}]')
print(f'y bounds : [{xyz[:,1].min():.4f},  {xyz[:,1].max():.4f}]')
print(f'z bounds : [{xyz[:,2].min():.4f},  {xyz[:,2].max():.4f}]')

## 2 — Volume grid coloured by Jacobian

Green = high-quality cells, red = poor quality (near inversion).  
The blue surface is the k=0 body wall.

In [ ]:
pl = view_geo(grid, jupyter=True)
pl.show()

## 3 — Per-layer Jacobian profile

Plot the minimum Jacobian as a function of wall-normal layer index.

In [ ]:
import matplotlib.pyplot as plt

jmins = jacobian_per_layer(grid)   # shape (nk-1,)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(jmins) + 1), jmins, marker='o', ms=4, lw=1.5)
ax.axhline(0, color='red', lw=0.8, ls='--', label='inversion threshold')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Minimum cell Jacobian per layer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Global jmin = {jmins.min():.4e}  at layer {jmins.argmin() + 1}')

## 4 — Jacobian full field on the PyVista mesh

Access the Jacobian array directly from the PyVista object.

In [ ]:
sg = geo_to_pyvista(grid, with_jacobian=True)
print('Cell arrays:', list(sg.cell_data.keys()))
jac = sg.cell_data['Jacobian']
print(f'Jacobian  min={jac.min():.4e}  max={jac.max():.4e}  mean={jac.mean():.4e}')

## 5 — Load and inspect the k=0 surface layer

In [ ]:
k0 = np.load(str(NPZ_PATH))['k0']   # shape (ni, nj, 3)
print(f'Surface shape : {k0.shape}')

radii = np.linalg.norm(k0, axis=-1)
print(f'Radius  mean={radii.mean():.4f}  std={radii.std():.4e}  '
      f'min={radii.min():.4f}  max={radii.max():.4f}')

## 6 — Surface layer coloured by radius from origin

In [ ]:
pl = view_surface(k0, jupyter=True, colorby='radius')
pl.show()

## 7 — Export to VTK for ParaView

Write `.vts` files that ParaView can open directly.

In [ ]:
write_vts('sphere.vts', grid)
write_surface_vts('sphere_k0.vts', k0)

print('Wrote sphere.vts and sphere_k0.vts — open in ParaView.')

## 8 — Slice at a specific k-layer

Inspect an individual wall-normal layer as a structured surface.

In [ ]:
k_slice = nk // 2    # midpoint layer
layer = grid[:, :, k_slice, :]   # shape (ni, nj, 3)

pl = view_surface(layer, jupyter=True, colorby='radius')
pl.add_title(f'Layer k={k_slice}')
pl.show()

## 9 — Spacing: surface clustering and wall-normal distributions

SHORE provides three spacing laws (`uniform`, `tanh`, `tanh2`) for controlling
node density along each direction.  Surface spacing is set at projection time;
volume spacing is set at march time.

In [ ]:
from shore.volume.spacing import build_i_coords, build_j_coords, build_k_steps

# --- Surface spacing examples ---

# Latitude: tanh clustering near the cap boundary
i_tanh = build_i_coords(40, 0.1, np.pi - 0.1, spacing="tanh", beta=4.0)

# Latitude: tanh2 clustering at both cap boundaries
i_tanh2 = build_i_coords(40, 0.1, np.pi - 0.1, spacing="tanh2", beta=4.0)

# Longitude: tanh2 clustering toward phi=0 (leading edge)
j_le = build_j_coords(60, spacing="tanh2", beta=5.0, cluster_lon=0.0)

print(f"Latitude tanh  — first spacing: {np.diff(i_tanh)[0]:.4f}, last: {np.diff(i_tanh)[-1]:.4f}")
print(f"Latitude tanh2 — mid spacing:   {np.diff(i_tanh2)[len(i_tanh2)//2]:.4f}, end: {np.diff(i_tanh2)[0]:.4f}")
print(f"Longitude LE   — spacing at LE: {np.diff(j_le)[np.argmin(np.abs(j_le))]:.4f}, at pi: {np.diff(j_le)[len(j_le)//2]:.4f}")

# --- Volume spacing examples ---

# Geometric (default)
k_geom = build_k_steps(20, ds=0.02, growth=1.10)

# Tanh: cluster near wall within fixed total thickness
k_tanh = build_k_steps(20, ds=0.02, spacing="tanh", beta=4.0, total_thickness=2.0)

# Tanh2: cluster at both wall and far-field
k_tanh2 = build_k_steps(20, ds=0.02, spacing="tanh2", beta=3.0, total_thickness=2.0)

print(f"\nGeometric — first: {k_geom[0]:.4f}, last: {k_geom[-1]:.4f}, total: {k_geom.sum():.3f}")
print(f"Tanh      — first: {k_tanh[0]:.4f}, last: {k_tanh[-1]:.4f}, total: {k_tanh.sum():.3f}")
print(f"Tanh2     — first: {k_tanh2[0]:.4f}, last: {k_tanh2[-1]:.4f}, total: {k_tanh2.sum():.3f}")

### Plot: step-size distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Surface spacing
ax = axes[0]
ax.semilogy(np.diff(i_tanh), 'o-', ms=3, label='i: tanh (wall cluster)')
ax.semilogy(np.diff(i_tanh2), 's-', ms=3, label='i: tanh2 (both ends)')
ax.semilogy(np.diff(j_le), '^-', ms=3, label='j: tanh2 (leading edge)')
ax.set_xlabel('index')
ax.set_ylabel('angular spacing (rad)')
ax.set_title('Surface spacing distributions')
ax.legend()
ax.grid(True, alpha=0.3)

# Volume spacing
ax = axes[1]
layers = np.arange(1, len(k_geom) + 1)
ax.semilogy(layers, k_geom, 'o-', ms=3, label='geometric')
ax.semilogy(layers, k_tanh, 's-', ms=3, label='tanh (wall cluster)')
ax.semilogy(layers, k_tanh2, '^-', ms=3, label='tanh2 (wall + far-field)')
ax.set_xlabel('layer k')
ax.set_ylabel('step size')
ax.set_title('Wall-normal step-size distributions')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Generate a grid with tanh volume spacing

Tanh spacing fixes the total wall-normal thickness and clusters layers near the
wall.  Compare the Jacobian profile with the geometric grid from section 3.

In [ ]:
TANH_GEO = Path('sphere_tanh.geo')

if not TANH_GEO.exists():
    k0 = np.load(str(NPZ_PATH))['k0']
    k_steps = build_k_steps(NK, ds=DS, spacing="tanh", beta=4.0, total_thickness=2.0)

    def _cb(k: int, jmin: float, step: float) -> None:
        print(f'  layer {k:3d}  jmin={jmin:.3e}  step={step:.3e}', end='\r')

    grid_tanh = march(k0, nk=NK, ds=DS, growth=GROWTH, steps=k_steps, layer_callback=_cb)
    write_geo(str(TANH_GEO), grid_tanh)
    print(f'\nGenerated {TANH_GEO}  shape={grid_tanh.shape[:3]}')
else:
    grid_tanh = read_geo(TANH_GEO)
    print(f'{TANH_GEO} already exists, skipping.')

grid_tanh = read_geo(TANH_GEO)

In [ ]:
# Compare Jacobian profiles: geometric vs tanh
jmins_geom = jacobian_per_layer(grid)
jmins_tanh = jacobian_per_layer(grid_tanh)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, len(jmins_geom) + 1), jmins_geom, 'o-', ms=4, lw=1.5, label='geometric')
ax.plot(range(1, len(jmins_tanh) + 1), jmins_tanh, 's-', ms=4, lw=1.5, label='tanh (beta=4)')
ax.axhline(0, color='red', lw=0.8, ls='--')
ax.set_xlabel('layer k')
ax.set_ylabel('jmin')
ax.set_title('Jacobian profile: geometric vs tanh spacing')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Geometric  global jmin = {jmins_geom.min():.4e}')
print(f'Tanh       global jmin = {jmins_tanh.min():.4e}')

### Generate a grid with surface clustering + tanh volume spacing

Combine longitude clustering (toward phi=0) with tanh wall-normal spacing.

In [ ]:
CLUSTERED_GEO = Path('sphere_clustered.geo')

if not CLUSTERED_GEO.exists():
    mesh = load_stl(STL_PATH)
    k0_cl = project_sphere_to_surface(
        mesh, ni=NI, nj=NJ,
        j_spacing='tanh2', j_beta=5.0, j_cluster_lon=0.0,
    )
    k_steps = build_k_steps(NK, ds=DS, spacing='tanh', beta=4.0, total_thickness=2.0)

    grid_cl = march(k0_cl, nk=NK, ds=DS, growth=GROWTH, steps=k_steps)
    write_geo(str(CLUSTERED_GEO), grid_cl)
    print(f'Generated {CLUSTERED_GEO}  shape={grid_cl.shape[:3]}')
else:
    grid_cl = read_geo(CLUSTERED_GEO)
    print(f'{CLUSTERED_GEO} already exists, skipping.')

pl = view_geo(grid_cl, jupyter=True, colorby='Jacobian')
pl.add_title('Surface clustering + tanh volume spacing')
pl.show()